| **Original Header**                                                         | **Converted to pet**                  | **Notes**                                                     |
|-----------------------------------------------------------------------------|---------------------------------------|---------------------------------------------------------------|
| Species                                                                     | pet_scientific_name                   |                                                               |
| Order                                                                       | [Removed]                             |                                                               |
| Suborder                                                                    | [Removed]                             |                                                               |
| Family                                                                      | pet_family                            |                                                               |
| Microhabitat                                                                | [Removed]                             |                                                               |
| Habitat type                                                                | pet_habitat                           |                                                               |
| Carapace color                                                              | pet_traits:turtle:carapace_colour     |                                                               |
| Dorsal colour                                                               | pet_traits:turtle:dorsal_colour       |                                                               |
| Dorsal pattern                                                              | pet_traits:turtle:dorsal_pattern      |                                                               |
| Pupil Shape                                                                 | [Removed]                             |                                                               |
| Maximum mass (g)                                                            | pet_max_weight                        | Convert to kg                                                 |
| Maximum length ("SVL", mm)/straight carapace length for turtles ("SCL", mm) | pet_max_length                        | Convert to cm; take greater of this column vs carapace length |
| Reproduction mode                                                           | [Removed]                             |                                                               |
| IUCN redlist assessment status (2024-2)                                     | pet_traits:iucn_status                | Append " (as of 2024)"                                        |
| IUCN population trend                                                       | pet_traits:iucn_trend                 |                                                               |
| English name                                                                | pet_vernacular_name                   |                                                               |
| Carapace length (adult, mm)                                                 | pet_max_length                        | Convert to cm; take greater of this column vs Maximum length  |
| Shell type (Hardshell/Softshell)                                            | pet_body_shape:turtle:shell_type      |                                                               |
| Plastron color                                                              | pet_traits:turtle:underside_colour    |                                                               |
| Neck retraction mechanism                                                   | [Removed]                             |                                                               |
| Number of toes/fingers (Forelimbs)                                          | pet_body_shape:turtle:no_of_toes_fore |                                                               |
| Number of toes/fingers (Hindlimbs)                                          | pet_body_shape:turtle:no_of_toes_hind |                                                               |
| Diet_chel                                                                   | pet_diet:main_type                    |                                                               |
| Detailed diet                                                               | pet_diet:remarks                      |                                                               |
| Active time                                                                 | [Removed]                             |                                                               |
| Lifespan (year)                                                             | pet_longevity                         |                                                               |
| Incubation period (day)                                                     | [Removed]                             |                                                               |
| Egg shape                                                                   | [Removed]                             |                                                               |
| Use and Trade (IUCN)                                                        | pet_comments                          | Append with "Used for "                                       |

Added: pet_is_native based on https://www.mybis.gov.my/pb/4503 and https://www.iucngisd.org/gisd/index.php Taxonomy: Animalia --> Chordata --> Reptilia --> Testudines & Location: South & Southeast Asia --> MALAYSIA

In [136]:
import re
import json
import pandas as pd
import numpy as np

In [137]:
input_file = "turtle_table_full.xlsx"
output_file = "pet_turtle.csv"

In [138]:
def clean_text(val, num = False):
    if pd.isna(val):
        return None
    val = str(val).strip()
    if num:
        match = re.search(r"[\d\.]+", val)
        return int(match.group()) if match else None
    return val if val else None


In [139]:
def extract_max_number(val):
    """
    Extract maximum numeric value from a cell
    """
    if pd.isna(val):
        return np.nan

    text = str(val).strip()
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if not nums:
        return np.nan

    nums = [float(x) for x in nums]
    #print(max(nums))
    return max(nums)

In [140]:
def safe_json_text(obj):
    def strip_empty(x):
        if isinstance(x, dict):
            cleaned = {k: strip_empty(v) for k, v in x.items()}
            return {k: v for k, v in cleaned.items() if v not in (None, "", {}, [])}
        return x

    cleaned = strip_empty(obj)
    if cleaned in (None, {}, [], ""):
        return None
    return json.dumps(cleaned, ensure_ascii=False)

In [141]:
df = pd.read_excel(input_file)

In [142]:
# Clean object columns
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(clean_text)

In [ ]:
# Core mapped columns
pet_scientific_name = df["Species"].apply(clean_text)
pet_family = df["Family"].apply(clean_text)
pet_habitat = df["Habitat type"].apply(clean_text)
pet_vernacular_name = df["English name"]
pet_genus = df["Genus"].apply(clean_text)
pet_family = df["Family"].apply(clean_text)
pet_longevity = df["Lifespan (year)"]
pet_longevity = pd.to_numeric(pet_longevity, errors='coerce')

In [ ]:
# Maximum mass (g) -> kg
mass_g = df["Maximum mass (g)"].apply(extract_max_number)
pet_max_weight = mass_g / 1000.0
pet_max_weight = pd.to_numeric(pet_max_weight, errors='coerce')

In [ ]:
len1_mm = df['Maximum length ("SVL", mm)/straight carapace length for turtles ("SCL", mm)'].apply(extract_max_number)
len2_mm = df["Carapace length (adult, mm)"].apply(extract_max_number)
pet_max_length = pd.to_numeric(pd.concat([len1_mm, len2_mm], axis=1).max(axis=1) / 10.0)

In [146]:
def build_pet_traits(row):
    iucn_status = clean_text(row.get("IUCN redlist assessment status (2024-2)"))
    if iucn_status:
        iucn_status = f"{iucn_status} (as of 2024)"

    obj = {
        "turtle": {
            "carapace_colour": clean_text(row.get("Carapace color")),
            "dorsal_colour": clean_text(row.get("Dorsal colour")),
            "dorsal_pattern": clean_text(row.get("Dorsal pattern")),
            "underside_colour": clean_text(row.get("Plastron color")),
        },
        "iucn_status": iucn_status,
        "iucn_trend": clean_text(row.get("IUCN population trend"))

    }
    return safe_json_text(obj)

In [147]:
def build_pet_diet(row):
    obj = {
        "main_type": clean_text(row.get("Diet_chel")),
        "remarks": clean_text(row.get("Detailed diet"))
    }
    return safe_json_text(obj)

In [148]:
def build_pet_body_shape(row):
    obj = {
        "turtle": {
            "shell_type": clean_text(row.get("Shell type (Hardshell/Softshell)")),
            "no_of_toes_fore": clean_text(row.get("Number of toes/fingers (Forelimbs)"), num=True),
            "no_of_toes_hind": clean_text(row.get("Number of toes/fingers (Hindlimbs)"), num=True),
        }
    }
    return safe_json_text(obj)

In [149]:
# pet_comments
def build_pet_comments(row):
    use_trade = clean_text(row.get("Use and Trade (IUCN)"))
    retraction = clean_text(row.get("Neck retraction mechanism"))
    active = clean_text(row.get("Active time"))
    return f"Used for {use_trade}. {retraction} retraction mechanism. Usually active at {active}."

In [150]:
pet_traits = df.apply(build_pet_traits, axis=1)
pet_body_shape = df.apply(build_pet_body_shape, axis=1)
pet_comments = df.apply(build_pet_comments, axis=1)
pet_diet = df.apply(build_pet_diet, axis=1)

In [ ]:
out = pd.DataFrame({
    "pet_id": pd.Series([pd.NA] * len(df), dtype="object"),
    "pet_scientific_name": pet_scientific_name,
    "pet_vernacular_name": pet_vernacular_name,
    "pet_genus": pet_genus,
    "pet_family": pet_family,
    "pet_body_shape": pet_body_shape,
    "pet_traits": pet_traits,
    "pet_max_length": pet_max_length,
    "pet_max_weight": pet_max_weight,
    "pet_longevity": pet_longevity,
    "pet_habitat": pet_habitat,
    "pet_temperature": pd.Series([None] * len(df), dtype="object"),
    "pet_ph_range": pd.Series([None] * len(df), dtype="object"),
    "pet_water_hardness": pd.Series([None] * len(df), dtype="object"),
    "pet_tank_size": pd.Series([None] * len(df), dtype="object"),
    "pet_migration_type": pd.Series([None] * len(df), dtype="object"),
    "pet_danger": pd.Series([None] * len(df), dtype="object"),
    "pet_is_native": pd.Series(['Not Native'] * len(df), dtype="object"),
    "pet_comments": pet_comments,
    "pet_aquarium": pd.Series([None] * len(df), dtype="object"),
    "pet_cost": pd.Series([None] * len(df), dtype="object"),
    "pet_image_ref": pd.Series([None] * len(df), dtype="object"),
    "pet_banned": pd.Series([None] * len(df), dtype="object"),
    "pet_invasive_risk": pd.Series([None] * len(df), dtype="object"),
    "pet_care_level": pd.Series([None] * len(df), dtype="object"),
    "pet_diet": pet_diet
})

In [152]:
# Assigning pet_id sequentially
out["pet_id"] = [f"{i}t" for i in range(1, 1 + len(out))]

In [153]:
# Assigning pet_tank_size based on max_length (to inch) * 10
def assign_tank_size(row):
    length = row["pet_max_length"]
    if pd.isna(length):
        return None
    return np.round((length / 2.54) * 10, decimals=-1)  # Round to nearest 10 gallons

In [154]:
# Assign tank size
out["pet_tank_size"] = out.apply(assign_tank_size, axis=1)

In [155]:
# Assigning 'Native' / 'Invasive'
native_species = [
    "Dermochelys coriacea",
    "Chelonia mydas",
    "Eretmochelys imbricata",
    "Lepidochelys olivacea",
    "Manouria impressa",
    "Manouria emys",
    "Indotestudo elongata",
    "Batagur affinis",
    "Heosemys grandis",
    "Batagur borneoensis",
    "Oritia borneensis",
    "Siebenrockiella crassicolli",
    "Heosemys annandalii",
    "Chitra chitra",
    "Amyda cartilaginea",
    "Notochelys platynota",
    "Malayemys macrocephala",
    "Cuora amboinensis",
    "Cyclemys dentata",
    "Heosemys spinosa",
    "Dogania subplana",
    "Pelochelys cantorii"
]

invasive_species = [
    "Trachemys scripta elegans",
    "Trachemys scripta scripta"
]

In [156]:
for idx, row in out.iterrows():
    sci_name = row["pet_scientific_name"]
    if sci_name in native_species:
        out.at[idx, "pet_is_native"] = "Native"
    elif sci_name in invasive_species:
        out.at[idx, "pet_is_native"] = "Invasive"


In [157]:
out.head(5)

,pet_id,pet_scientific_name,pet_vernacular_name,pet_genus,pet_family,pet_body_shape,pet_traits,pet_max_length,pet_max_weight,pet_longevity,...,pet_danger,pet_is_native,pet_comments,pet_aquarium,pet_cost,pet_image_ref,pet_banned,pet_invasive_risk,pet_care_level,pet_diet
0,1t,Acanthochelys macrocephala,"Pantanal swamp turtle, Big-headed pantanal swa...",Acanthochelys,Chelidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark brown, bl...",29.5,2.895,8.2,...,None,Not Native,"Used for Pets/display animals, horticulture. S...",None,None,None,None,None,None,"{""main_type"": ""Carnivorous"", ""remarks"": ""Inver..."
1,2t,Acanthochelys pallidipectoris,Chaco side-necked turtle,Acanthochelys,Chelidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Brown, olive c...",29.5,0.872,8.2,...,None,Not Native,"Used for Pets/display animals, horticulture. S...",None,None,None,None,None,None,"{""main_type"": ""Carnivorous"", ""remarks"": ""Amphi..."
2,3t,Acanthochelys radiolata,Brazilian radiolated swamp turtle,Acanthochelys,Chelidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Grayish brown""...",20.0,1.126,8.2,...,None,Not Native,"Used for Pets/display animals, horticulture. S...",None,None,None,None,None,None,"{""main_type"": ""Carnivorous"", ""remarks"": ""Aquat..."
3,4t,Acanthochelys spixii,"Black spiny-necked turtle, Spix’s sideneck turtle",Acanthochelys,Chelidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark brown, ol...",18.0,0.759,8.2,...,None,Not Native,"Used for Pets/display animals, horticulture. S...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Snails..."
4,5t,Actinemys pallida,"Southern pacific pond turtle, Southwestern pon...",Actinemys,Emydidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Yellowish brow...",21.3,0.325,70,...,None,Not Native,"Used for Pets/display animals, Food-human. Hid...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Amphib..."


In [158]:
out[out["pet_is_native"] == "Native"]

,pet_id,pet_scientific_name,pet_vernacular_name,pet_genus,pet_family,pet_body_shape,pet_traits,pet_max_length,pet_max_weight,pet_longevity,...,pet_danger,pet_is_native,pet_comments,pet_aquarium,pet_cost,pet_image_ref,pet_banned,pet_invasive_risk,pet_care_level,pet_diet
6,7t,Amyda cartilaginea,Asiatic softshell turtle,Amyda,Trionychidae,"{""turtle"": {""shell_type"": ""Softshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Grayish green,...",80.0,202.000000,7.3,...,None,Native,Used for Food-human. Hidden-necked retraction ...,None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Amphib..."
13,14t,Batagur affinis,Southern river terrapin,Batagur,Geoemydidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark grayish ,...",62.5,38.000000,21.7,...,None,Native,Used for Food-human. Hidden-necked retraction ...,None,None,None,None,None,None,"{""main_type"": ""Herbivorous"", ""remarks"": ""Inver..."
15,16t,Batagur borneoensis,Painted terrapin,Batagur,Geoemydidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Light yellow ,...",76.0,50.848000,6.8,...,None,Native,"Used for Food-human, Pets. Hidden-necked retra...",None,None,None,None,None,None,"{""main_type"": ""Herbivorous"", ""remarks"": ""Leave..."
39,40t,Chelonia mydas,"Green turtle, green sea turtle",Chelonia,Cheloniidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Olive color, b...",153.0,380.000000,75,...,None,Native,Used for Food-human. Hidden-necked retraction ...,None,None,None,None,None,None,"{""main_type"": ""Herbivorous"", ""remarks"": ""Omniv..."
53,54t,Chitra chitra,Asian narrow-headed softshell turtle,Chitra,Trionychidae,"{""turtle"": {""shell_type"": ""Softshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Yellow, light ...",150.0,254.000000,140,...,None,Native,"Used for Food-human, Pets. Hidden-necked retra...",None,None,None,None,None,None,"{""main_type"": ""Carnivorous"", ""remarks"": ""Fish;..."
60,61t,Cuora amboinensis,Southeast asian box turtle,Cuora,Geoemydidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark black"", ""...",25.0,2.261000,38.2,...,None,Native,"Used for Establishing ex-situ production, Food...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Invert..."
76,77t,Cyclemys dentata,Asian leaf turtle,Cyclemys,Geoemydidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark brown"", ""...",24.0,1.447128,14.8,...,None,Native,"Used for Food-human, Pets. Hidden-necked retra...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Vegeta..."
86,87t,Dermochelys coriacea,Leatherback,Dermochelys,Dermochelyidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark brown, sl...",270.0,950.000000,30,...,None,Native,"Used for Other chemicals, Food-human, Food-ani...",None,None,None,None,None,None,"{""main_type"": ""Carnivorous"", ""remarks"": ""Inver..."
87,88t,Dogania subplana,Malayan soft-shelled turtle,Dogania,Trionychidae,"{""turtle"": {""shell_type"": ""Softshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark olive col...",35.0,10.988052,18.9,...,None,Native,"Used for Food-human, Pets. Hidden-necked retra...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Omnivo..."
113,114t,Eretmochelys imbricata,"Hawksbill turtle, Hawksbill sea turtle",Eretmochelys,Cheloniidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Dark green bro...",114.0,140.000000,45,...,None,Native,"Used for Handicrafts, jewellery, etc.. Hidden-...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Sea In..."


In [159]:
out[out["pet_is_native"] == "Invasive"]

,pet_id,pet_scientific_name,pet_vernacular_name,pet_genus,pet_family,pet_body_shape,pet_traits,pet_max_length,pet_max_weight,pet_longevity,...,pet_danger,pet_is_native,pet_comments,pet_aquarium,pet_cost,pet_image_ref,pet_banned,pet_invasive_risk,pet_care_level,pet_diet
352,353t,Trachemys scripta scripta,Yellow-bellied slider,Trachemys,Emydidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Olive color, b...",32.8,3.51333,75,...,None,Invasive,"Used for Pets/display animals, horticulture, F...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Crusta..."
353,354t,Trachemys scripta elegans,Red-eared Slider,Trachemys,Emydidae,"{""turtle"": {""shell_type"": ""Hardshell"", ""no_of_...","{""turtle"": {""carapace_colour"": ""Olive color, b...",32.8,3.51333,75,...,None,Invasive,"Used for Pets/display animals, horticulture, F...",None,None,None,None,None,None,"{""main_type"": ""Omnivorous"", ""remarks"": ""Crusta..."


In [160]:
out.to_csv(output_file, index=False, encoding="utf-8")
print(f'Saved to {output_file}')

Saved to pet_turtle.csv
